# DS4400 Homework 3: Classification
**Ariv Ahuja**

In [ ]:
# Imports and setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, roc_curve, auc)
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

In [ ]:
# Load the SPAMBASE dataset
feature_names = [
    'word_freq_make', 'word_freq_address', 'word_freq_all', 'word_freq_3d',
    'word_freq_our', 'word_freq_over', 'word_freq_remove', 'word_freq_internet',
    'word_freq_order', 'word_freq_mail', 'word_freq_receive', 'word_freq_will',
    'word_freq_people', 'word_freq_report', 'word_freq_addresses', 'word_freq_free',
    'word_freq_business', 'word_freq_email', 'word_freq_you', 'word_freq_credit',
    'word_freq_your', 'word_freq_font', 'word_freq_000', 'word_freq_money',
    'word_freq_hp', 'word_freq_hpl', 'word_freq_george', 'word_freq_650',
    'word_freq_lab', 'word_freq_labs', 'word_freq_telnet', 'word_freq_857',
    'word_freq_data', 'word_freq_415', 'word_freq_85', 'word_freq_technology',
    'word_freq_1999', 'word_freq_parts', 'word_freq_pm', 'word_freq_direct',
    'word_freq_cs', 'word_freq_meeting', 'word_freq_original', 'word_freq_project',
    'word_freq_re', 'word_freq_edu', 'word_freq_table', 'word_freq_conference',
    'char_freq_;', 'char_freq_(', 'char_freq_[', 'char_freq_!', 'char_freq_$',
    'char_freq_#', 'capital_run_length_average', 'capital_run_length_longest',
    'capital_run_length_total'
]
col_names = feature_names + ['spam']
data = pd.read_csv('spambase/spambase.data', header=None, names=col_names)
X = data.drop('spam', axis=1).values
y = data['spam'].values
print(f'Dataset shape: {X.shape}, Spam: {y.sum()}, Ham: {(1-y).sum()}')

# 75/25 train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f'Training set: {X_train.shape[0]} samples, Testing set: {X_test.shape[0]} samples')

## Problem 1: Logistic Regression

In [ ]:
# 1.1 Train logistic regression and evaluate on test set
lr = LogisticRegression(max_iter=10000, random_state=42)
lr.fit(X_train_scaled, y_train)
y_pred = lr.predict(X_test_scaled)
y_proba = lr.predict_proba(X_test_scaled)[:, 1]

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print('Confusion Matrix:')
print(cm)
print(f'  TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}')

# Metrics
acc = accuracy_score(y_test, y_pred)
err = 1 - acc
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print(f'\nAccuracy:  {acc:.4f}')
print(f'Error:     {err:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall:    {rec:.4f}')
print(f'F1 Score:  {f1:.4f}')

In [ ]:
# 1.2 Feature coefficients
coefs = lr.coef_[0]
coef_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': coefs})
coef_df = coef_df.reindex(coef_df['Coefficient'].abs().sort_values(ascending=False).index)

print('Top 10 features by absolute coefficient:')
for _, row in coef_df.head(10).iterrows():
    sign = '+' if row['Coefficient'] > 0 else '-'
    print(f"  [{sign}] {row['Feature']}: {row['Coefficient']:.4f}")

print('\nTop 5 POSITIVELY correlated with SPAM:')
pos = coef_df[coef_df['Coefficient'] > 0].head(5)
for _, row in pos.iterrows():
    print(f"  {row['Feature']}: {row['Coefficient']:.4f}")

print('\nTop 5 NEGATIVELY correlated with SPAM:')
neg = coef_df[coef_df['Coefficient'] < 0].head(5)
for _, row in neg.iterrows():
    print(f"  {row['Feature']}: {row['Coefficient']:.4f}")

In [ ]:
# 1.3 Vary decision threshold
thresholds_1 = [0.25, 0.5, 0.75, 0.9]
print(f'{"Threshold":<12} {"Accuracy":<12} {"Precision":<12} {"Recall":<12}')
print('-' * 48)
for t in thresholds_1:
    y_pred_t = (y_proba >= t).astype(int)
    a = accuracy_score(y_test, y_pred_t)
    p = precision_score(y_test, y_pred_t, zero_division=0)
    r = recall_score(y_test, y_pred_t, zero_division=0)
    print(f'{t:<12.2f} {a:<12.4f} {p:<12.4f} {r:<12.4f}')

## Problem 2: Gradient Descent for Logistic Regression

In [ ]:
# Custom gradient descent implementation for logistic regression
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))

def cross_entropy_loss(X, y, theta):
    m = len(y)
    h = sigmoid(X @ theta)
    h = np.clip(h, 1e-10, 1 - 1e-10)
    return -(1/m) * (y @ np.log(h) + (1 - y) @ np.log(1 - h))

def logistic_gradient_descent(X, y, lr, n_iter, record_at=None):
    m, n = X.shape
    theta = np.zeros(n)
    losses = {}
    for i in range(1, n_iter + 1):
        h = sigmoid(X @ theta)
        gradient = (1/m) * (X.T @ (h - y))
        theta -= lr * gradient
        if record_at and i in record_at:
            losses[i] = cross_entropy_loss(X, y, theta)
    return theta, losses

# Add bias column
X_train_b = np.c_[np.ones(X_train_scaled.shape[0]), X_train_scaled]
X_test_b = np.c_[np.ones(X_test_scaled.shape[0]), X_test_scaled]

learning_rates = [0.01, 0.1, 1.0]
record_iters = [10, 50, 100]

# Cross-entropy loss at various iterations
print('Cross-Entropy Loss at Various Iterations:')
print(f'{"LR":<8} {"Iter 10":<14} {"Iter 50":<14} {"Iter 100":<14}')
print('-' * 50)

gd_results = {}
for alpha in learning_rates:
    theta, losses = logistic_gradient_descent(X_train_b, y_train, alpha, 100, record_at=set(record_iters))
    gd_results[alpha] = (theta, losses)
    print(f'{alpha:<8.2f} {losses.get(10, float("nan")):<14.6f} {losses.get(50, float("nan")):<14.6f} {losses.get(100, float("nan")):<14.6f}')

In [ ]:
# Metrics at 100 iterations for each learning rate
print('Metrics at 100 Iterations - Training Set:')
print(f'{"LR":<8} {"Accuracy":<12} {"Precision":<12} {"Recall":<12} {"F1":<12}')
print('-' * 56)
for alpha in learning_rates:
    theta = gd_results[alpha][0]
    yp = (sigmoid(X_train_b @ theta) >= 0.5).astype(int)
    print(f'{alpha:<8.2f} {accuracy_score(y_train, yp):<12.4f} {precision_score(y_train, yp):<12.4f} {recall_score(y_train, yp):<12.4f} {f1_score(y_train, yp):<12.4f}')

print('\nMetrics at 100 Iterations - Testing Set:')
print(f'{"LR":<8} {"Accuracy":<12} {"Precision":<12} {"Recall":<12} {"F1":<12}')
print('-' * 56)
for alpha in learning_rates:
    theta = gd_results[alpha][0]
    yp = (sigmoid(X_test_b @ theta) >= 0.5).astype(int)
    print(f'{alpha:<8.2f} {accuracy_score(y_test, yp):<12.4f} {precision_score(y_test, yp):<12.4f} {recall_score(y_test, yp):<12.4f} {f1_score(y_test, yp):<12.4f}')

# sklearn comparison
print('\nsklearn Logistic Regression (comparison):')
yp_tr = lr.predict(X_train_scaled)
yp_te = lr.predict(X_test_scaled)
print(f'  Train: Acc={accuracy_score(y_train, yp_tr):.4f}, Prec={precision_score(y_train, yp_tr):.4f}, Rec={recall_score(y_train, yp_tr):.4f}, F1={f1_score(y_train, yp_tr):.4f}')
print(f'  Test:  Acc={accuracy_score(y_test, yp_te):.4f}, Prec={precision_score(y_test, yp_te):.4f}, Rec={recall_score(y_test, yp_te):.4f}, F1={f1_score(y_test, yp_te):.4f}')

## Problem 3: Comparing Classifiers

In [ ]:
# 3.1 kNN cross-validation for k
k_values = [1, 3, 5, 7, 9, 11, 15, 21, 31, 51]
print(f'{"k":<6} {"Accuracy":<12} {"Error":<12} {"Precision":<12} {"Recall":<12}')
print('-' * 54)
best_k = None
best_error = float('inf')
cv_results_knn = []
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    cv_acc = cross_val_score(knn, X_train_scaled, y_train, cv=5, scoring='accuracy').mean()
    cv_prec = cross_val_score(knn, X_train_scaled, y_train, cv=5, scoring='precision').mean()
    cv_rec = cross_val_score(knn, X_train_scaled, y_train, cv=5, scoring='recall').mean()
    cv_err = 1 - cv_acc
    cv_results_knn.append((k, cv_acc, cv_err, cv_prec, cv_rec))
    print(f'{k:<6} {cv_acc:<12.4f} {cv_err:<12.4f} {cv_prec:<12.4f} {cv_rec:<12.4f}')
    if cv_err < best_error:
        best_error = cv_err
        best_k = k

print(f'\nBest k = {best_k} (CV error = {best_error:.4f})')

# Plot kNN CV error vs k
plt.figure(figsize=(6, 4))
plt.plot([r[0] for r in cv_results_knn], [r[2] for r in cv_results_knn], 'bo-')
plt.xlabel('k')
plt.ylabel('CV Error')
plt.title('kNN: Cross-Validation Error vs k')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figures/knn_cv.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# 3.2 Train all 3 classifiers and compare
lr_model = LogisticRegression(max_iter=10000, random_state=42)
lda_model = LinearDiscriminantAnalysis()
knn_model = KNeighborsClassifier(n_neighbors=best_k)

lr_model.fit(X_train_scaled, y_train)
lda_model.fit(X_train_scaled, y_train)
knn_model.fit(X_train_scaled, y_train)

models = {'Logistic Regression': lr_model, 'LDA': lda_model, f'kNN (k={best_k})': knn_model}

print('Training Set Metrics:')
print(f'{"Model":<25} {"Accuracy":<12} {"Error":<12} {"Precision":<12} {"Recall":<12}')
print('-' * 73)
for name, model in models.items():
    yp = model.predict(X_train_scaled)
    a = accuracy_score(y_train, yp)
    print(f'{name:<25} {a:<12.4f} {1-a:<12.4f} {precision_score(y_train, yp):<12.4f} {recall_score(y_train, yp):<12.4f}')

print('\nTesting Set Metrics:')
print(f'{"Model":<25} {"Accuracy":<12} {"Error":<12} {"Precision":<12} {"Recall":<12}')
print('-' * 73)
for name, model in models.items():
    yp = model.predict(X_test_scaled)
    a = accuracy_score(y_test, yp)
    print(f'{name:<25} {a:<12.4f} {1-a:<12.4f} {precision_score(y_test, yp):<12.4f} {recall_score(y_test, yp):<12.4f}')

In [ ]:
# 3.3 ROC Curve with sklearn
y_proba_lr = lr_model.predict_proba(X_test_scaled)[:, 1]
fpr_pkg, tpr_pkg, _ = roc_curve(y_test, y_proba_lr)
auc_pkg = auc(fpr_pkg, tpr_pkg)
print(f'AUC (sklearn): {auc_pkg:.4f}')

plt.figure(figsize=(6, 5))
plt.plot(fpr_pkg, tpr_pkg, 'b-', label=f'ROC (AUC = {auc_pkg:.4f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Logistic Regression (sklearn)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figures/roc_sklearn.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# 3.4 Manual ROC Curve
manual_thresholds = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
manual_fprs = []
manual_tprs = []
print(f'{"Threshold":<12} {"FPR":<12} {"TPR":<12}')
print('-' * 36)
for t in manual_thresholds:
    y_pred_t = (y_proba_lr >= t).astype(int)
    tn = np.sum((y_pred_t == 0) & (y_test == 0))
    fp = np.sum((y_pred_t == 1) & (y_test == 0))
    fn = np.sum((y_pred_t == 0) & (y_test == 1))
    tp = np.sum((y_pred_t == 1) & (y_test == 1))
    fpr_m = fp / (fp + tn) if (fp + tn) > 0 else 0
    tpr_m = tp / (tp + fn) if (tp + fn) > 0 else 0
    manual_fprs.append(fpr_m)
    manual_tprs.append(tpr_m)
    print(f'{t:<12.1f} {fpr_m:<12.4f} {tpr_m:<12.4f}')

plt.figure(figsize=(6, 5))
plt.plot(fpr_pkg, tpr_pkg, 'b-', label=f'sklearn ROC (AUC = {auc_pkg:.4f})')
plt.plot(manual_fprs, manual_tprs, 'ro-', label='Manual ROC (11 thresholds)')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison: sklearn vs Manual')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figures/roc_comparison.pdf', bbox_inches='tight')
plt.show()

## Problem 4: Cross Validation

In [ ]:
# 4.1 Custom k-fold cross-validation implementation
def custom_k_fold_cv(X, y, model_class, k, **model_kwargs):
    n = len(y)
    indices = np.arange(n)
    np.random.seed(42)
    np.random.shuffle(indices)
    fold_size = n // k
    errors = []

    for i in range(k):
        start = i * fold_size
        end = start + fold_size if i < k - 1 else n
        val_idx = indices[start:end]
        train_idx = np.concatenate([indices[:start], indices[end:]])

        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        # Standardize within each fold
        sc = StandardScaler()
        X_tr_s = sc.fit_transform(X_tr)
        X_val_s = sc.transform(X_val)

        model = model_class(**model_kwargs)
        model.fit(X_tr_s, y_tr)
        y_pred_v = model.predict(X_val_s)
        fold_error = 1 - accuracy_score(y_val, y_pred_v)
        errors.append(fold_error)

    return errors, np.mean(errors)

# 4.2 Run CV for logistic regression and LDA with k=5 and k=10
for k in [5, 10]:
    print(f'k = {k}:')
    lr_errors, lr_avg = custom_k_fold_cv(X, y, LogisticRegression, k, max_iter=10000, random_state=42)
    lda_errors, lda_avg = custom_k_fold_cv(X, y, LinearDiscriminantAnalysis, k)

    print(f'  Logistic Regression fold errors: {[round(e, 4) for e in lr_errors]}')
    print(f'  Logistic Regression avg validation error: {lr_avg:.4f}')
    print(f'  LDA fold errors: {[round(e, 4) for e in lda_errors]}')
    print(f'  LDA avg validation error: {lda_avg:.4f}')
    print()